# Cognitive–behavioral CCA analysis

Canonical correlation analysis (CCA) between cognitive and behavioral test blocks. Helpers at the top support **full cohort** and **high p-tau217** subsets via `COHORTS_TO_RUN`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"pdf.fonttype": 42, "ps.fonttype": 42})

import seaborn as sns
from sklearn.cross_decomposition import CCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import pearsonr

import sys

sys.path.append("/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/")
from clinical_data_analysis.statistical_analysis.settings import *



In [ ]:
DATA_PATH = "/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/Data/fightAD_general_data_table.xlsx"
SAVE_DIR = "/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/figures/"

## Helper functions and analysis parameters


In [ ]:
# --- Parameters (tune for speed vs stability) ---
COHORTS_TO_RUN = ["full", "high_ptau"]  # "full", "high_ptau", or both

N_SPLITS = 5
N_PERM_IN_SAMPLE = 1000
N_PERM_CV = 200  # increase to 1000+ for final publication runs
N_BOOT_CI = 1000
RANDOM_STATE = 42
PUBLICATION_TOP_N = 8

COHORT_LABELS = {
    "full": "full cohort",
    "high_ptau": f"high p-tau217 cohort (>{HIGH_TAU_217_THR})",
}


def _can_corrs(x_scores, y_scores):
    return np.array([pearsonr(x_scores[:, k], y_scores[:, k])[0] for k in range(x_scores.shape[1])])


def _fisher_mean_r(rs):
    rs = np.asarray(rs, dtype=float)
    rs = rs[np.isfinite(rs)]
    if len(rs) == 0:
        return np.nan
    rs = np.clip(rs, -0.999999, 0.999999)
    return float(np.tanh(np.mean(np.arctanh(rs))))


def cv_can1_corr(Xz, Yz, n_splits=5, random_state=42):
    """Cross-validated correlation between Can1 scores (held-out evaluation)."""
    from sklearn.model_selection import KFold

    kf = KFold(n_splits=n_splits, shuffle=True, random_state=random_state)
    fold_rs = []

    for train_idx, test_idx in kf.split(Xz):
        cca_cv = CCA(n_components=1)
        cca_cv.fit(Xz[train_idx], Yz[train_idx])
        Xc_te, Yc_te = cca_cv.transform(Xz[test_idx], Yz[test_idx])
        fold_rs.append(pearsonr(Xc_te[:, 0], Yc_te[:, 0])[0])

    return _fisher_mean_r(fold_rs)


def permutation_pvalue_first_mode_in_sample(Xz, Yz, observed_r, n_perm=1000, random_state=42):
    """Permutation p-value for in-sample Can1 (optimistic; included for comparison)."""
    rng_local = np.random.default_rng(random_state)
    null_rs = np.zeros(n_perm)

    for i in range(n_perm):
        perm_idx = rng_local.permutation(Xz.shape[0])
        Y_perm = Yz[perm_idx, :]

        cca_perm = CCA(n_components=1)
        Xc_p, Yc_p = cca_perm.fit_transform(Xz, Y_perm)
        null_rs[i] = np.abs(pearsonr(Xc_p[:, 0], Yc_p[:, 0])[0])

    return (np.sum(null_rs >= np.abs(observed_r)) + 1) / (n_perm + 1)


def permutation_pvalue_cv_can1(Xz, Yz, observed_r_cv, n_perm=200, n_splits=5, random_state=42):
    """Permutation p-value for cross-validated Can1 correlation."""
    rng_local = np.random.default_rng(random_state)
    null_rs = np.zeros(n_perm)

    for i in range(n_perm):
        perm_idx = rng_local.permutation(Xz.shape[0])
        Y_perm = Yz[perm_idx, :]
        null_rs[i] = np.abs(cv_can1_corr(Xz, Y_perm, n_splits=n_splits, random_state=random_state + i + 1))

    return (np.sum(null_rs >= np.abs(observed_r_cv)) + 1) / (n_perm + 1)


def bootstrap_loading_ci(feature_df, can_scores, n_boot=1000, random_state=42):
    rng_local = np.random.default_rng(random_state)
    n = len(feature_df)
    feats = feature_df.columns
    boot = np.zeros((n_boot, len(feats)))

    for b in range(n_boot):
        idx = rng_local.integers(0, n, size=n)
        x_boot = feature_df.iloc[idx].reset_index(drop=True)
        s_boot = pd.Series(can_scores[idx])
        boot[b, :] = x_boot.corrwith(s_boot).values

    lo = np.nanpercentile(boot, 2.5, axis=0)
    hi = np.nanpercentile(boot, 97.5, axis=0)
    return pd.DataFrame({"ci_low": lo, "ci_high": hi}, index=feats)


def build_c_b_df(df, cognitive_tests, behavioral_tests):
    return df[cognitive_tests + behavioral_tests + [TAU_217]].copy()


def select_cohort(c_b_df, cohort):
    """Return (analysis_df without tau, tau_series) for the requested cohort."""
    if cohort not in COHORT_LABELS:
        raise ValueError(f"Unknown cohort {cohort!r}. Use one of {list(COHORT_LABELS)}")

    tau_series = c_b_df[TAU_217].copy()

    if cohort == "full":
        analysis_df = c_b_df.drop(columns=[TAU_217])
    elif cohort == "high_ptau":
        mask = c_b_df[TAU_217] > HIGH_TAU_217_THR
        analysis_df = c_b_df.loc[mask].drop(columns=[TAU_217])
        tau_series = tau_series.loc[mask]
    else:
        raise ValueError(f"Unhandled cohort: {cohort}")

    return analysis_df, tau_series


def prepare_cca_matrices(analysis_df, cognitive_tests, behavioral_tests):
    X = analysis_df[cognitive_tests].copy()
    Y = analysis_df[behavioral_tests].copy()
    data = pd.concat([X, Y], axis=1).dropna()
    X_arr = data[cognitive_tests].values
    Y_arr = data[behavioral_tests].values
    X_z = StandardScaler().fit_transform(X_arr)
    Y_z = StandardScaler().fit_transform(Y_arr)
    n_components = min(len(cognitive_tests), len(behavioral_tests))
    return X_z, Y_z, data, n_components


def fit_cca(X_z, Y_z, n_components):
    cca = CCA(n_components=n_components)
    X_c, Y_c = cca.fit_transform(X_z, Y_z)

    can_corrs = []
    can_pvals = []
    for k in range(n_components):
        r, p = pearsonr(X_c[:, k], Y_c[:, k])
        can_corrs.append(r)
        can_pvals.append(p)

    can_df = pd.DataFrame(
        {"Canonical correlation": can_corrs, "p-value": can_pvals},
        index=[f"Comp{k+1}" for k in range(n_components)],
    )
    return cca, X_c, Y_c, can_corrs, can_pvals, can_df


def compute_can1_structure(
    c_b_clean_df,
    X_c,
    Y_c,
    cognitive_tests,
    behavioral_tests,
    n_boot=N_BOOT_CI,
    random_state=RANDOM_STATE,
    top_n=PUBLICATION_TOP_N,
):
    cog_struct = c_b_clean_df[cognitive_tests].corrwith(
        pd.Series(X_c[:, 0], index=c_b_clean_df.index)
    )
    cog_cross = c_b_clean_df[cognitive_tests].corrwith(
        pd.Series(Y_c[:, 0], index=c_b_clean_df.index)
    )
    beh_struct = c_b_clean_df[behavioral_tests].corrwith(
        pd.Series(Y_c[:, 0], index=c_b_clean_df.index)
    )
    beh_cross = c_b_clean_df[behavioral_tests].corrwith(
        pd.Series(X_c[:, 0], index=c_b_clean_df.index)
    )

    cog_ci = bootstrap_loading_ci(
        c_b_clean_df[cognitive_tests], X_c[:, 0], n_boot=n_boot, random_state=random_state
    )
    beh_ci = bootstrap_loading_ci(
        c_b_clean_df[behavioral_tests], Y_c[:, 0], n_boot=n_boot, random_state=random_state
    )

    if top_n is None:
        top_cog_idx = cog_struct.index
        top_beh_idx = beh_struct.index
    else:
        top_n_eff = min(top_n, len(cognitive_tests), len(behavioral_tests))
        top_cog_idx = cog_struct.abs().sort_values(ascending=False).head(top_n_eff).index
        top_beh_idx = beh_struct.abs().sort_values(ascending=False).head(top_n_eff).index

    cog_plot = pd.DataFrame(
        {
            "Structure (corr with Cognitive Can1)": cog_struct.loc[top_cog_idx],
            "Cross-loading (corr with Behavioral Can1)": cog_cross.loc[top_cog_idx],
            "ci_low": cog_ci.loc[top_cog_idx, "ci_low"],
            "ci_high": cog_ci.loc[top_cog_idx, "ci_high"],
        }
    ).sort_values("Structure (corr with Cognitive Can1)")

    beh_plot = pd.DataFrame(
        {
            "Structure (corr with Behavioral Can1)": beh_struct.loc[top_beh_idx],
            "Cross-loading (corr with Cognitive Can1)": beh_cross.loc[top_beh_idx],
            "ci_low": beh_ci.loc[top_beh_idx, "ci_low"],
            "ci_high": beh_ci.loc[top_beh_idx, "ci_high"],
        }
    ).sort_values("Structure (corr with Behavioral Can1)")

    return {
        "cog_struct": cog_struct,
        "cog_cross": cog_cross,
        "beh_struct": beh_struct,
        "beh_cross": beh_cross,
        "cog_plot": cog_plot,
        "beh_plot": beh_plot,
    }


def run_cca_inference(
    X_z,
    Y_z,
    X_c,
    Y_c,
    data,
    tau_series,
    cognitive_tests,
    behavioral_tests,
    cohort,
    n_splits=N_SPLITS,
    n_perm_in_sample=N_PERM_IN_SAMPLE,
    n_perm_cv=N_PERM_CV,
    n_boot=N_BOOT_CI,
    random_state=RANDOM_STATE,
):
    can_corrs_arr = _can_corrs(X_c, Y_c)
    r1 = can_corrs_arr[0]
    p1_pearson = pearsonr(X_c[:, 0], Y_c[:, 0])[1]

    p1_perm_in = permutation_pvalue_first_mode_in_sample(
        X_z, Y_z, observed_r=r1, n_perm=n_perm_in_sample, random_state=random_state
    )
    r1_cv = cv_can1_corr(X_z, Y_z, n_splits=n_splits, random_state=random_state)
    p1_perm_cv = permutation_pvalue_cv_can1(
        X_z,
        Y_z,
        observed_r_cv=r1_cv,
        n_perm=n_perm_cv,
        n_splits=n_splits,
        random_state=random_state,
    )

    c_b_clean_df = data[cognitive_tests + behavioral_tests].copy()
    tau_aligned = tau_series.loc[c_b_clean_df.index]
    structure = compute_can1_structure(
        c_b_clean_df,
        X_c,
        Y_c,
        cognitive_tests,
        behavioral_tests,
        n_boot=n_boot,
        random_state=random_state,
        top_n=PUBLICATION_TOP_N,
    )

    return {
        "cohort": cohort,
        "cohort_label": COHORT_LABELS[cohort],
        "n_cca": len(c_b_clean_df),
        "can_corrs_arr": can_corrs_arr,
        "r1": r1,
        "p1_pearson": p1_pearson,
        "p1_perm_in": p1_perm_in,
        "r1_cv": r1_cv,
        "p1_perm_cv": p1_perm_cv,
        "tau_aligned": tau_aligned,
        "structure": structure,
    }


def plot_canonical_correlations_bar(can_corrs, title_suffix=""):
    plt.figure(figsize=(6, 4))
    plt.bar(range(1, len(can_corrs) + 1), can_corrs)
    plt.xlabel("Canonical component")
    plt.ylabel("Correlation")
    plt.title(f"Canonical correlations between cognitive and behavioral blocks{title_suffix}")
    plt.tight_layout()
    plt.show()


def plot_loadings_heatmaps(cca, cognitive_tests, behavioral_tests, n_components, title_suffix=""):
    cog_loadings = pd.DataFrame(
        cca.x_loadings_,
        index=cognitive_tests,
        columns=[f"Can{j+1}" for j in range(n_components)],
    )
    behav_loadings = pd.DataFrame(
        cca.y_loadings_,
        index=behavioral_tests,
        columns=[f"Can{j+1}" for j in range(n_components)],
    )

    plt.figure(figsize=(10, max(4, 0.4 * len(cognitive_tests))))
    sns.heatmap(cog_loadings, annot=True, fmt=".2f", cmap="coolwarm", center=0, rasterized=False)
    plt.title(f"CCA loadings - cognitive tests{title_suffix}")
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(10, max(4, 0.4 * len(behavioral_tests))))
    sns.heatmap(behav_loadings, annot=True, fmt=".2f", cmap="coolwarm", center=0, rasterized=False)
    plt.title(f"CCA loadings - behavioral tests{title_suffix}")
    plt.tight_layout()
    plt.show()

    return cog_loadings, behav_loadings


def plot_can1_all_features(data, X_c, Y_c, cognitive_tests, behavioral_tests, title_suffix=""):
    from matplotlib.patches import Patch

    c_b_clean_df = data[cognitive_tests + behavioral_tests].copy()
    can1_x = X_c[:, 0]
    can1_y = Y_c[:, 0]

    cog_vs_x = c_b_clean_df[cognitive_tests].corrwith(pd.Series(can1_x, index=c_b_clean_df.index))
    cog_vs_y = c_b_clean_df[cognitive_tests].corrwith(pd.Series(can1_y, index=c_b_clean_df.index))
    behav_vs_y = c_b_clean_df[behavioral_tests].corrwith(pd.Series(can1_y, index=c_b_clean_df.index))
    behav_vs_x = c_b_clean_df[behavioral_tests].corrwith(pd.Series(can1_x, index=c_b_clean_df.index))

    fig, axes = plt.subplots(
        1,
        2,
        figsize=(14, max(6, 0.45 * max(len(cognitive_tests), len(behavioral_tests)))),
        sharex=True,
    )

    cog_plot_df = pd.DataFrame(
        {
            "Within-group (Cognitive Can1)": cog_vs_x,
            "Cross-group (Behavioral Can1)": cog_vs_y,
        }
    ).sort_values("Within-group (Cognitive Can1)")
    cog_plot_df.plot(kind="barh", ax=axes[0], color=["tab:blue", "tab:orange"], alpha=0.85)
    axes[0].axvline(0, color="black", linewidth=0.8)
    axes[0].set_title("Cognitive features and Can1")
    axes[0].set_xlabel("Pearson r")
    axes[0].set_ylabel("Cognitive features")

    behav_plot_df = pd.DataFrame(
        {
            "Within-group (Behavioral Can1)": behav_vs_y,
            "Cross-group (Cognitive Can1)": behav_vs_x,
        }
    ).sort_values("Within-group (Behavioral Can1)")
    behav_plot_df.plot(kind="barh", ax=axes[1], color=["tab:green", "tab:red"], alpha=0.85)
    axes[1].axvline(0, color="black", linewidth=0.8)
    axes[1].set_title("Behavioral features and Can1")
    axes[1].set_xlabel("Pearson r")
    axes[1].set_ylabel("Behavioral features")

    axes[0].legend_.remove()
    axes[1].legend_.remove()
    legend_handles = [
        Patch(facecolor="tab:blue", alpha=0.85, label="Cognitive features vs Cognitive Can1 (within-group)"),
        Patch(facecolor="tab:orange", alpha=0.85, label="Cognitive features vs Behavioral Can1 (cross-group)"),
        Patch(facecolor="tab:green", alpha=0.85, label="Behavioral features vs Behavioral Can1 (within-group)"),
        Patch(facecolor="tab:red", alpha=0.85, label="Behavioral features vs Cognitive Can1 (cross-group)"),
    ]
    fig.legend(
        handles=legend_handles,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.965),
        ncol=2,
        frameon=False,
        fontsize=9,
    )
    fig.suptitle(f"Can1 interpretation: within-group vs cross-group feature relationships{title_suffix}", y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.93])
    plt.show()


def plot_cca_publication_panel(results, X_c, Y_c, save_dir, cohort):
    """4-panel publication figure (A-D)."""
    can_corrs_arr = results["can_corrs_arr"]
    r1 = results["r1"]
    p1_pearson = results["p1_pearson"]
    p1_perm_in = results["p1_perm_in"]
    r1_cv = results["r1_cv"]
    p1_perm_cv = results["p1_perm_cv"]
    tau_aligned = results["tau_aligned"]
    cog_plot = results["structure"]["cog_plot"]
    beh_plot = results["structure"]["beh_plot"]
    cohort_label = results["cohort_label"]

    fig = plt.figure(figsize=(16, 12))
    gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.42)

    axA = fig.add_subplot(gs[0, 0])
    bar_colors = ["tab:blue" if i == 0 else "lightsteelblue" for i in range(len(can_corrs_arr))]
    axA.bar(range(1, len(can_corrs_arr) + 1), can_corrs_arr, color=bar_colors, edgecolor="black", linewidth=0.5)
    axA.axhline(0, color="black", linewidth=0.8)
    axA.set_xticks(range(1, len(can_corrs_arr) + 1))
    axA.set_xlabel("Canonical mode")
    axA.set_ylabel("Canonical correlation (r)")
    axA.set_title("A. Canonical correlations")
    axA.text(
        0.02,
        0.98,
        f"Can1 (in-sample): r={r1:.2f}\nPearson p={p1_pearson:.3g}\nPerm (in-sample) p={p1_perm_in:.3g}\n\n"
        f"Can1 (CV): r={r1_cv:.2f}\nPerm (CV) p={p1_perm_cv:.3g}",
        transform=axA.transAxes,
        va="top",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.9, edgecolor="0.8"),
    )

    axB = fig.add_subplot(gs[0, 1])
    ptau217_clipped = np.clip(tau_aligned.values, None, 1)
    sc = axB.scatter(
        X_c[:, 0],
        Y_c[:, 0],
        c=ptau217_clipped,
        cmap="viridis",
        s=22,
        alpha=0.5,
        edgecolor="k",
        linewidth=0.3,
        vmin=np.nanmin(ptau217_clipped),
        vmax=1.0,
    )
    sns.regplot(
        x=X_c[:, 0],
        y=Y_c[:, 0],
        scatter=False,
        line_kws={"color": "black", "linewidth": 1.5},
        ax=axB,
    )
    axB.set_xlabel("Cognitive Can1 scores")
    axB.set_ylabel("Behavioral Can1 scores")
    axB.set_title("B. Can1 score association")
    plt.colorbar(sc, ax=axB, label="p-tau217 (saturated at 1)")
    axB.text(
        0.02,
        0.98,
        f"in-sample r={r1:.2f}, p={p1_pearson:.3g}\nCV r={r1_cv:.2f}",
        transform=axB.transAxes,
        va="top",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.9, edgecolor="0.8"),
    )

    axC = fig.add_subplot(gs[1, 0])
    y_cog = np.arange(len(cog_plot))
    axC.barh(y_cog, cog_plot["Structure (corr with Cognitive Can1)"], color="tab:blue", alpha=0.85, label="Structure")
    axC.barh(
        y_cog,
        cog_plot["Cross-loading (corr with Behavioral Can1)"],
        color="tab:orange",
        alpha=0.75,
        label="Cross-loading",
    )
    xerr_cog = np.vstack(
        [
            cog_plot["Structure (corr with Cognitive Can1)"] - cog_plot["ci_low"],
            cog_plot["ci_high"] - cog_plot["Structure (corr with Cognitive Can1)"],
        ]
    )
    axC.errorbar(
        cog_plot["Structure (corr with Cognitive Can1)"],
        y_cog,
        xerr=xerr_cog,
        fmt="none",
        ecolor="black",
        elinewidth=1.2,
        capsize=3,
    )
    axC.set_yticks(y_cog)
    axC.set_yticklabels(cog_plot.index)
    axC.axvline(0, color="black", linewidth=0.8)
    axC.set_xlabel("Correlation with Can1")
    axC.set_title("C. Cognitive features: top structure/cross-loadings\n(95% bootstrap CI on structure)")
    axC.legend(frameon=False, fontsize=9)

    axD = fig.add_subplot(gs[1, 1])
    y_beh = np.arange(len(beh_plot))
    axD.barh(y_beh, beh_plot["Structure (corr with Behavioral Can1)"], color="tab:blue", alpha=0.85, label="Structure")
    axD.barh(
        y_beh,
        beh_plot["Cross-loading (corr with Cognitive Can1)"],
        color="tab:orange",
        alpha=0.75,
        label="Cross-loading",
    )
    xerr_beh = np.vstack(
        [
            beh_plot["Structure (corr with Behavioral Can1)"] - beh_plot["ci_low"],
            beh_plot["ci_high"] - beh_plot["Structure (corr with Behavioral Can1)"],
        ]
    )
    axD.errorbar(
        beh_plot["Structure (corr with Behavioral Can1)"],
        y_beh,
        xerr=xerr_beh,
        fmt="none",
        ecolor="black",
        elinewidth=1.2,
        capsize=3,
    )
    axD.set_yticks(y_beh)
    axD.set_yticklabels(beh_plot.index, fontsize=9)
    axD.yaxis.tick_right()
    axD.tick_params(axis="y", labelright=True, labelleft=False)
    axD.axvline(0, color="black", linewidth=0.8)
    axD.set_xlabel("Correlation with Can1")
    axD.set_title("D. Behavioral features: top structure/cross-loadings\n(95% bootstrap CI on structure)")
    axD.legend(frameon=False, fontsize=9)

    fig.suptitle(f"CCA summary panel ({cohort_label})", fontsize=16, y=0.995)
    fig.tight_layout(rect=[0, 0, 1, 0.98])
    out_path = f"{save_dir}/CCA_publication_panel_{cohort}.pdf"
    fig.savefig(out_path, dpi=300, bbox_inches="tight")
    plt.show()
    print(f"Saved: {out_path}")


def plot_explained_variance(X_c, Y_c, X_z, Y_z, n_components, title_suffix=""):
    X_c_var = np.var(X_c, axis=0)
    Y_c_var = np.var(Y_c, axis=0)
    X_var = np.var(X_z, axis=0).sum()
    Y_var = np.var(Y_z, axis=0).sum()
    explained_var_X = X_c_var / X_var
    explained_var_Y = Y_c_var / Y_var
    can_explained_var = (explained_var_X + explained_var_Y) / 2

    plt.figure(figsize=(10, 5))
    plt.bar(range(1, n_components + 1), can_explained_var)
    plt.xlabel("Canonical component")
    plt.ylabel("Explained variance (average of X and Y)")
    plt.title(f"Explained variance of CCA components (average for X and Y){title_suffix}")
    plt.tight_layout()
    plt.show()
    return can_explained_var



## Load and filter data


In [ ]:
df = pd.read_excel(DATA_PATH, sheet_name="1")

df.columns = df.loc[0]
df = df.drop(0)
df = df.iloc[:, :-2]

df.rename(
    columns={
        "#": "ID",
        "תאריך בדיקות דם": "blood_test_date",
        "שעת לקיחת דם": "blood_taken_time",
        "מספר מנות": "num_of_injections",
        "גיל הפסקת מחזור": "menopause",
        "אסטרוגן": "estorgen",
        "Frontal memory score (#1)- כמה מילים זכר/ה ברמז הראשון. נקודה אם זכר/ה רמז ראשון ושתי נקודות אם לא.": "Frontal memory score (#1)",
        "Hipocampal memory score (#2)- כמה מילים לא זכר/ה בכלל": "Hipocampal memory score (#2)",
        "p-tau_ new_#1": "p-tau_new_#1",
        "ביקורת אחרי שנה": "checkup after a year",
        "חיסון": "vaccine",
        "תרופות": "medications",
        "הערות": "notes",
        "Number_of_risk_faktors": "Number_of_risk_factors",
        "P-tau217": "p-tau217",
        "צרבונין": "Cerebonin",
        "Spatial ": "Spatial",
    },
    inplace=True,
)
df = df.apply(pd.to_numeric, errors="ignore")
df = df.drop(df[df["ID"].isin(OUTLIERS)].index)



In [ ]:
df = df.drop(df[df["ID"].isin(OUTLIERS)].index)
df = df[df[MOCA_COL] != "?"]



In [ ]:
negative_columns = [
    "Negative MOCA_score",
    "Negative Executive",
    "Negative Spatial",
    "Negative Attention_and_concentration",
    "Negative Memory",
    "Negative Language",
    "Negative Orientation",
    "Negative IST",
]

df[negative_columns] = df[
    ["MOCA_score", "Executive", "Spatial", "Attention_and_concentration", "Memory", "Language", "Orientation", "IST"]
].apply(lambda x: x.astype(float) * -1)

df[negative_columns].head(1)



In [ ]:
cognitive_tests = negative_columns + [
    "Frontal memory score (#1)",
    "Hipocampal memory score (#2)",
    "Trail_B_(second)",
]
behavioral_tests = [
    "Mild_behavioral_Impairment_Checklist",
    "interest, motivation, and drive",
    "mood or anxiety symptoms",
    "Impulse control",
    "following societal norms",
    "strongly held beliefs and sensory",
    "iADL",
]



In [ ]:
TAU_COLUMNS = [TAU_217]
COGNITIVE_COLUMN = "Trail_B_(second)"
BEHAVIORAL_COLUMN = "Mild_behavioral_Impairment_Checklist"

print(f"Before: {df.shape}")
df = df.dropna(subset=[COGNITIVE_COLUMN, BEHAVIORAL_COLUMN, TAU_217])
print(f"After: {df.shape}")
df = df[df[MOCA_COL] >= 19]
print(f"After removing samples with MOCA score below 18: {df.shape}")



In [ ]:
df[COGNITIVE_COLUMN] = df[COGNITIVE_COLUMN].astype(int)
df[MCI] = (df[COGNITIVE_COLUMN] < MCI_THR).astype(int)
df[BEHAVIORAL_COLUMN] = df[BEHAVIORAL_COLUMN].astype(int)



In [ ]:
c_b_df = build_c_b_df(df, cognitive_tests, behavioral_tests)
print(f"c_b_df shape (all subjects with tau): {c_b_df.shape}")
print(f"High p-tau217 rows: {(c_b_df[TAU_217] > HIGH_TAU_217_THR).sum()}")



## Run CCA per cohort

Set `COHORTS_TO_RUN = ['full']`, `['high_ptau']`, or both.


In [ ]:
for cohort in COHORTS_TO_RUN:
    print("\n" + "=" * 72)
    print(f"CCA analysis — {COHORT_LABELS[cohort]}")
    print("=" * 72)

    analysis_df, tau_series = select_cohort(c_b_df, cohort)
    print(f"N in cohort (before complete-case drop): {len(analysis_df)}")

    X_z, Y_z, data, n_components = prepare_cca_matrices(
        analysis_df, cognitive_tests, behavioral_tests
    )
    print(f"N after dropping NaNs in cognitive + behavioral blocks: {data.shape[0]}")

    cca, X_c, Y_c, can_corrs, can_pvals, can_df = fit_cca(X_z, Y_z, n_components)
    display(can_df)

    title_suffix = f" ({cohort})"

    plot_canonical_correlations_bar(can_corrs, title_suffix=title_suffix)
    plot_loadings_heatmaps(
        cca, cognitive_tests, behavioral_tests, n_components, title_suffix=title_suffix
    )
    plot_can1_all_features(
        data, X_c, Y_c, cognitive_tests, behavioral_tests, title_suffix=title_suffix
    )

    results = run_cca_inference(
        X_z,
        Y_z,
        X_c,
        Y_c,
        data,
        tau_series,
        cognitive_tests,
        behavioral_tests,
        cohort=cohort,
    )
    print(
        f"Can1 in-sample r={results['r1']:.3f}, p={results['p1_pearson']:.3g}; "
        f"CV r={results['r1_cv']:.3f}; perm(CV) p={results['p1_perm_cv']:.3g}"
    )

    plot_cca_publication_panel(results, X_c, Y_c, SAVE_DIR, cohort)
    plot_explained_variance(X_c, Y_c, X_z, Y_z, n_components, title_suffix=title_suffix)

